# **Gemma Co-Radiologist: DDA Agent + Proficiency Benchmark**### Declarative DataFlow Agent × MedGemma × MedSigLIP/FAISS Retrieval**Competition submission** — benchmarks the DDA agent against Vanilla MedGemma and GPT-4o on the CheXpert validation set using the **Proficiency Index** metric.#### Architecture```Query + Image      │      ▼[MedSigLIP Retrieval] → Top-K similar CXRs with labels      │      ▼[Few-Shot Analysis] → MedGemma with KNN context      │      ▼[Verification] → Validated CXRReview JSON (14 labels)      │      ▼[Proficiency Index] → % correct labels per image```

### **Installations**

In [ ]:
# ── Install athena_dda SDK + dependencies ──
# Upload athena_dda-0.1.0-py3-none-any.whl to your Kaggle dataset
!pip install -q /kaggle/input/athena-dda-sdk/athena_dda-0.1.0-py3-none-any.whl
!pip install -q transformers accelerate huggingface_hub faiss-cpu pydantic openai scikit-learn matplotlib
print("✓ All packages installed")

### **SDK Imports from `athena_dda`**

In [ ]:
import os, json, re, time, asyncio, base64
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from datetime import datetime
from typing import Dict, List, Optional, Any, Callable
from pydantic import BaseModel
from enum import Enum

# ── DDA SDK (pip-installed) ──
from athena_dda import (
    ToolRegistry, ToolSchema, ToolType, ArgExtractorType,
    ContextBank, LLMValueExtractor,
    GroundedArgumentFiller,
    DataFlowEngine,
    GoalKeyResolver,
    DeclarativeDataFlowAgent, ConfigurableOrchestrator,
    SessionStore, SESSION_STORE,
    run_medgemma, load_medgemma, set_pipeline,
)

print(f"✓ athena_dda v{__import__('athena_dda').__version__} imported")
print(f"  DeclarativeDataFlowAgent: {DeclarativeDataFlowAgent}")
print(f"  ToolRegistry: {ToolRegistry}")

### **HuggingFace Authentication**

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

secrets  = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("✓ HuggingFace login successful")

### **Load MedGemma + MedSigLIP Models**

In [ ]:
import torch
from transformers import pipeline as hf_pipeline, AutoProcessor, AutoModel
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# ── MedGemma (for analysis) ──
medgemma_pipe = hf_pipeline(
    "image-text-to-text",
    model="google/medgemma-1.5-4b-it",
    device=device,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
)
set_pipeline(medgemma_pipe)  # Register with athena_dda
print("✓ MedGemma loaded")

# ── MedSigLIP (for retrieval) ──
SIGLIP_MODEL = "google/medsiglip-448"
siglip_processor = AutoProcessor.from_pretrained(SIGLIP_MODEL)
siglip_model     = AutoModel.from_pretrained(SIGLIP_MODEL, torch_dtype=torch.float16).to(device).eval()
print("✓ MedSigLIP loaded")

### **Constants & Data Loading**

In [ ]:
# ── Paths ──
MEDSIG_LIP_SIZE  = (448, 448)
EMBED_BATCH_SIZE = 128
NUM_WORKERS      = 4
KAGGLE_ROOT      = "/kaggle/input/datasets/ashery/chexpert"
LOCAL_INDEX_PATH  = "/kaggle/working/chexpert_train.index"
LOCAL_META_PATH   = "/kaggle/working/chexpert_train_meta.npz"

# ── CheXpert label columns (the 14 competition labels) ──
LABEL_COLS = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly",
    "Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
    "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]

# ── Load CSVs ──
TRAIN_CSV = os.path.join(KAGGLE_ROOT, "train.csv")
VALID_CSV = os.path.join(KAGGLE_ROOT, "valid.csv")
train_df  = pd.read_csv(TRAIN_CSV)
valid_df  = pd.read_csv(VALID_CSV)

train_paths = [
    os.path.join(KAGGLE_ROOT, p.replace("CheXpert-v1.0-small/", ""))
    for p in train_df["Path"].tolist()
]
valid_paths = [
    os.path.join(KAGGLE_ROOT, p.replace("CheXpert-v1.0-small/", ""))
    for p in valid_df["Path"].tolist()
]

print(f"Train images: {len(train_paths)}")
print(f"Valid images: {len(valid_paths)}")
print(f"Label columns: {len(LABEL_COLS)}")

### **FAISS Index Builder**From retrieval-notebook.ipynb — builds MedSigLIP embeddings + FAISS index.

In [ ]:
import faiss
import logging, random

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)


class CheXpertDataset(Dataset):
    def __init__(self, paths, processor, size=(448, 448)):
        self.paths     = paths
        self.processor = processor
        self.size      = size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            img    = Image.open(path).convert("RGB").resize(self.size, Image.BICUBIC)
            inputs = self.processor(images=img, return_tensors="pt")
            return {k: v.squeeze(0) for k, v in inputs.items()}, path, True
        except Exception:
            img    = Image.new("RGB", self.size)
            inputs = self.processor(images=img, return_tensors="pt")
            return {k: v.squeeze(0) for k, v in inputs.items()}, path, False


def build_faiss_index(
    local_paths, index_save_path=LOCAL_INDEX_PATH,
    metadata_save_path=LOCAL_META_PATH,
    batch_size=EMBED_BATCH_SIZE, num_workers=NUM_WORKERS,
):
    dataset    = CheXpertDataset(local_paths, siglip_processor, size=MEDSIG_LIP_SIZE)
    dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=num_workers,
                            pin_memory=True, prefetch_factor=2)

    path_store, failed_paths = {}, []
    faiss_index = None
    faiss_id = 0

    for batch_inputs, batch_paths, batch_valid in tqdm(dataloader, desc="Embedding", unit="batch"):
        batch_inputs = {k: v.to(device, non_blocking=True) for k, v in batch_inputs.items()}
        with torch.no_grad():
            outputs = siglip_model.get_image_features(**batch_inputs)
            emb = outputs if torch.is_tensor(outputs) else getattr(outputs, "pooler_output", outputs[0])
        emb = emb / emb.norm(dim=-1, keepdim=True)
        emb = emb.cpu().float().numpy().astype("float32")

        valid_indices = [i for i, v in enumerate(batch_valid) if v.item()]
        failed_paths.extend([batch_paths[i] for i, v in enumerate(batch_valid) if not v.item()])
        if not valid_indices:
            continue

        valid_embs = emb[valid_indices]
        if faiss_index is None:
            faiss_index = faiss.IndexFlatIP(valid_embs.shape[1])
            logger.info(f"FAISS index initialised | dim={valid_embs.shape[1]}")

        faiss_index.add(valid_embs)
        for i in valid_indices:
            rel_path = batch_paths[i].replace(KAGGLE_ROOT + "/", "")
            path_store[faiss_id] = rel_path
            faiss_id += 1

    if faiss_index is None or faiss_index.ntotal == 0:
        raise RuntimeError("No embeddings built.")

    faiss.write_index(faiss_index, index_save_path)
    np.savez_compressed(metadata_save_path, path_store=path_store)
    logger.info(f"Done | indexed={faiss_index.ntotal} | skipped={len(failed_paths)}")
    return faiss_index, path_store


# ── Build index (10k sample for speed) ──
random.seed(42)
train_paths_subset = random.sample(train_paths, 10_000)
index, path_store = build_faiss_index(train_paths_subset)
print(f"\nTotal indexed: {index.ntotal} | Dim: {index.d}")

### **Retrieval Functions**MedSigLIP embedding + FAISS search + label description builder.

In [ ]:
def build_label_description(row: pd.Series) -> str:
    present, absent, uncertain = [], [], []
    for col in LABEL_COLS:
        val = row.get(col, np.nan)
        if pd.isna(val): continue
        val = float(val)
        if val == 1.0:    present.append(col)
        elif val == 0.0:  absent.append(col)
        elif val == -1.0: uncertain.append(col)

    parts = []
    if present:   parts.append(f"Present: {', '.join(present)}.")
    if uncertain: parts.append(f"Uncertain: {', '.join(uncertain)}.")
    if absent:    parts.append(f"Absent: {', '.join(absent)}.")
    if not parts: parts.append("No findings documented.")

    meta = []
    for field in ["Sex", "Age", "Frontal/Lateral", "AP/PA"]:
        if field in row and not pd.isna(row.get(field)):
            meta.append(f"{field}: {row[field]}")
    if meta:
        parts.insert(0, " | ".join(meta) + ".")
    return " ".join(parts)


def build_description_lookup(csv_paths):
    lookup = {}
    for csv_path in csv_paths:
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            key = row["Path"].replace("CheXpert-v1.0-small/", "")
            lookup[key] = build_label_description(row)
    return lookup


def embed_query_image(image_path: str) -> np.ndarray:
    img    = Image.open(image_path).convert("RGB").resize(MEDSIG_LIP_SIZE, Image.BICUBIC)
    inputs = siglip_processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = siglip_model.get_image_features(**inputs)
        emb = outputs if torch.is_tensor(outputs) else getattr(outputs, "pooler_output", outputs[0])
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu().float().numpy().astype("float32")


def faiss_retrieve(query_vec, k=3):
    scores, indices = index.search(query_vec.reshape(1, -1), k)
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        if idx == -1: continue
        rel_path = path_store[idx]
        results.append({
            "rank": rank, "score": round(float(score), 4),
            "path": rel_path,
            "description": description_lookup.get(rel_path, "No description."),
        })
    return results


# Build lookup
description_lookup = build_description_lookup([TRAIN_CSV, VALID_CSV])
print(f"✓ Description lookup built | {len(description_lookup)} entries")

### **Pydantic Schemas & Prompt Skills**CXRReview with all 14 CheXpert labels. Maps directly to LABEL_COLS for benchmarking.

In [ ]:
class Ternary(str, Enum):
    yes = "yes"; no = "no"; unclear = "unclear"

class CXRReview(BaseModel):
    no_finding: Ternary
    enlarged_cardiomediastinum: Ternary
    cardiomegaly: Ternary
    lung_opacity: Ternary
    lung_lesion: Ternary
    edema: Ternary
    consolidation: Ternary
    pneumonia: Ternary
    atelectasis: Ternary
    pneumothorax: Ternary
    pleural_effusion: Ternary
    pleural_other: Ternary
    fracture: Ternary
    support_devices: Ternary
    verification_confidence: str = "medium"
    verification_notes: str = ""

# Map CXRReview fields → LABEL_COLS index (exact 14-field correspondence)
CXRREVIEW_TO_LABEL = {
    "no_finding": "No Finding",
    "enlarged_cardiomediastinum": "Enlarged Cardiomediastinum",
    "cardiomegaly": "Cardiomegaly",
    "lung_opacity": "Lung Opacity",
    "lung_lesion": "Lung Lesion",
    "edema": "Edema",
    "consolidation": "Consolidation",
    "pneumonia": "Pneumonia",
    "atelectasis": "Atelectasis",
    "pneumothorax": "Pneumothorax",
    "pleural_effusion": "Pleural Effusion",
    "pleural_other": "Pleural Other",
    "fracture": "Fracture",
    "support_devices": "Support Devices",
}


def cxr_review_to_binary(review_dict: dict) -> list:
    """Convert CXRReview dict to binary label vector aligned with LABEL_COLS."""
    result = []
    for field, label in CXRREVIEW_TO_LABEL.items():
        val = review_dict.get(field, "no")
        result.append(1 if str(val).lower() == "yes" else 0)
    return result


CXR_ANALYSIS_PROMPT = """You are an expert radiologist. Analyze this chest X-ray.
For each finding, respond with exactly "yes", "no", or "unclear".

Return ONLY a JSON object with these exact fields:
{
    "no_finding": "yes/no/unclear",
    "enlarged_cardiomediastinum": "yes/no/unclear",
    "cardiomegaly": "yes/no/unclear",
    "lung_opacity": "yes/no/unclear",
    "lung_lesion": "yes/no/unclear",
    "edema": "yes/no/unclear",
    "consolidation": "yes/no/unclear",
    "pneumonia": "yes/no/unclear",
    "atelectasis": "yes/no/unclear",
    "pneumothorax": "yes/no/unclear",
    "pleural_effusion": "yes/no/unclear",
    "pleural_other": "yes/no/unclear",
    "fracture": "yes/no/unclear",
    "support_devices": "yes/no/unclear",
    "verification_confidence": "low/medium/high",
    "verification_notes": "brief explanation"
}"""


print(f"✓ CXRReview schema: {len(CXRREVIEW_TO_LABEL)} labels mapped to LABEL_COLS")

### **Tool Implementations**Core radiology pipeline tools using real MedSigLIP retrieval + MedGemma analysis.

In [ ]:
def _retrieve_similar_images(patient_image: str, image_type: str, k: int = 5) -> str:
    """Retrieve similar images using MedSigLIP + FAISS."""
    if os.path.exists(patient_image):
        query_vec  = embed_query_image(patient_image)
        query_type = "image"
    else:
        inputs = siglip_processor(text=[f"{image_type} chest radiograph"], return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            outputs = siglip_model.get_text_features(**inputs)
            emb = outputs if torch.is_tensor(outputs) else getattr(outputs, "pooler_output", outputs[0])
        emb = emb / emb.norm(dim=-1, keepdim=True)
        query_vec = emb.cpu().float().numpy().astype("float32")
        query_type = "text"

    rel_path = patient_image.replace(KAGGLE_ROOT + "/", "")
    query_desc = description_lookup.get(rel_path, "No description.")
    results = faiss_retrieve(query_vec, k=k)

    return json.dumps({
        "query_image": patient_image, "query_description": query_desc,
        "image_type": image_type, "query_type": query_type,
        "k": len(results), "similar_images": results,
    }, indent=2)


def _few_shot_image_analysis(knn_images: str, patient_image: str) -> str:
    """Analyze image using KNN examples as few-shot context."""
    try:
        knn_data = json.loads(knn_images) if isinstance(knn_images, str) else knn_images
        similar  = knn_data.get("similar_images", [])
    except (json.JSONDecodeError, TypeError):
        similar = []

    context_parts = []
    for r in similar[:5]:
        context_parts.append(f"- Similar case (score={r.get('score', 0)}): {r.get('description', '')}")
    context_str = "\n".join(context_parts) if context_parts else "No similar cases found."

    prompt = (
        f"You are an expert radiologist performing chest X-ray analysis.\n\n"
        f"SIMILAR CASES FROM DATABASE:\n{context_str}\n\n"
        f"{CXR_ANALYSIS_PROMPT}\n\n"
        f"Analyze the patient's chest X-ray and return ONLY the JSON object."
    )

    content_blocks = [{"type": "text", "text": prompt}]
    if os.path.exists(patient_image):
        content_blocks.insert(0, {"type": "image", "image": Image.open(patient_image).convert("RGB")})

    response = run_medgemma(
        messages=[{"role": "user", "content": content_blocks}],
        max_new_tokens=512, temperature=0.1,
    )
    return response


def _verify_few_shot_image_analysis(analysis: str, patient_image: str) -> str:
    """Verify and correct the analysis with a second LLM pass + Pydantic validation."""
    prompt = (
        f"You are a senior radiologist verifying a preliminary chest X-ray report.\n\n"
        f"PRELIMINARY ANALYSIS:\n{analysis}\n\n"
        f"Review the image again and correct any errors. "
        f"{CXR_ANALYSIS_PROMPT}"
    )

    content_blocks = [{"type": "text", "text": prompt}]
    if os.path.exists(patient_image):
        content_blocks.insert(0, {"type": "image", "image": Image.open(patient_image).convert("RGB")})

    response = run_medgemma(
        messages=[{"role": "user", "content": content_blocks}],
        max_new_tokens=512, temperature=0.1,
    )

    # Try to parse and validate with Pydantic
    try:
        s = response.find("{")
        e = response.rfind("}")
        if s != -1 and e > s:
            data = json.loads(response[s:e+1])
            review = CXRReview(**data)
            return review.model_dump_json()
    except Exception:
        pass

    return response


print("✓ 3 core tools defined: retrieve, analyze, verify")

### **Tool Registration**Register tools into the DDA registry with collision-free DAG validation.

In [ ]:
# ── Radiology Configuration ──
RADIOLOGY_SYSTEM_PROMPT = (
    "You are an expert Co-Radiologist AI assistant. "
    "Provide structured CXR findings based on the clinical data."
)

RADIOLOGY_GOAL_EXAMPLES = (
    "EXAMPLES:\n"
    "Query: Analyze this chest X-ray -> verified_analysis\n"
    "Query: Find similar images -> knn_images\n"
    "Query: Verify the analysis -> verified_analysis\n"
    "Query: What are the findings? -> verified_analysis\n"
)

# ── Build registry ──
radiology_registry = ToolRegistry()

radiology_registry.register(
    name="retrieve_similar_images",
    description="Retrieve similar CXR images from FAISS index using MedSigLIP embeddings",
    tool_type=ToolType.RETRIEVAL,
    arg_sources={"patient_image": "patient_image_input", "image_type": "image_type_input"},
    arg_extractor={"name": "retrieve_similar_images", "arguments": {
        "patient_image": {"type": ArgExtractorType.LLM, "description": "Path to patient image"},
        "image_type": {"type": ArgExtractorType.LLM, "description": "xray|ct|mri"},
    }},
    func=_retrieve_similar_images,
    output_keys={"knn_images": "string"},
)

radiology_registry.register(
    name="few_shot_image_analysis",
    description="Analyze CXR using retrieved similar images as few-shot context",
    tool_type=ToolType.COMPUTATION,
    arg_sources={"knn_images": "knn_images", "patient_image": "patient_image_input"},
    arg_extractor={"name": "few_shot_image_analysis", "arguments": {
        "knn_images": {"type": ArgExtractorType.LLM, "description": "KNN retrieval results"},
        "patient_image": {"type": ArgExtractorType.LLM, "description": "Path to patient image"},
    }},
    func=_few_shot_image_analysis,
    output_keys={"few_shot_analysis": "string"},
)

radiology_registry.register(
    name="verify_few_shot_image_analysis",
    description="Verify and validate the CXR analysis with Pydantic schema enforcement",
    tool_type=ToolType.COMPUTATION,
    arg_sources={"analysis": "few_shot_analysis", "patient_image": "patient_image_input"},
    arg_extractor={"name": "verify_few_shot_image_analysis", "arguments": {
        "analysis": {"type": ArgExtractorType.LLM, "description": "Preliminary analysis"},
        "patient_image": {"type": ArgExtractorType.LLM, "description": "Path to patient image"},
    }},
    func=_verify_few_shot_image_analysis,
    output_keys={"verified_analysis": "string"},
)

# Validate DAG
engine = DataFlowEngine(radiology_registry, verbose=True)
plan = engine.resolve_execution_plan({"patient_image_input", "image_type_input"}, "verified_analysis")
print(f"\nExecution plan for verified_analysis: {plan}")
print(f"Graph stats: {engine.get_graph_stats()}")

### **Benchmark Metrics**Proficiency Index + AUROC per label from retrieval-notebook.ipynb.

In [ ]:
from sklearn.metrics import roc_auc_score


def proficiency_index(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Per-image proficiency index: mean % correct labels across all images."""
    correct = (y_true == y_pred).astype(float)
    row_scores = correct.mean(axis=1)
    return float(row_scores.mean())


def auroc_per_label(y_true: np.ndarray, y_prob: np.ndarray) -> dict:
    """Per-label AUROC. Ignores uncertain (-1) labels."""
    aurocs = {}
    for j, label in enumerate(LABEL_COLS):
        mask = y_true[:, j] != -1
        if mask.sum() < 10:
            aurocs[label] = None
            continue
        y_bin = (y_true[mask, j] == 1).astype(int)
        try:
            aurocs[label] = round(roc_auc_score(y_bin, y_prob[mask, j]), 4)
        except ValueError:
            aurocs[label] = None
    return aurocs


def zero_shot_vote(similar_images: list) -> dict:
    """Majority vote across top-K retrieved image labels."""
    votes = {col: 0 for col in LABEL_COLS}
    for result in similar_images:
        desc = result.get("description", "")
        if "Present:" in desc:
            present_section = desc.split("Present:")[1].split(".")[0]
            present_labels  = [l.strip() for l in present_section.split(",")]
            for label in present_labels:
                if label in votes:
                    votes[label] += 1
    k = max(len(similar_images), 1)
    return {label: int(count > k / 2) for label, count in votes.items()}


def parse_cxr_json(response: str) -> dict:
    """Extract CXR JSON from LLM response, return empty dict on failure."""
    try:
        s = response.find("{")
        e = response.rfind("}")
        if s != -1 and e > s:
            data = json.loads(response[s:e+1])
            return data
    except (json.JSONDecodeError, TypeError):
        pass
    return {}


def get_ground_truth_vector(path: str) -> list:
    """Get binary ground truth vector for a validation image."""
    rel_path = path.replace(KAGGLE_ROOT + "/", "")
    row = valid_df[valid_df["Path"].str.replace("CheXpert-v1.0-small/", "") == rel_path]
    if row.empty:
        return None
    row = row.iloc[0]
    return [1 if float(row.get(col, 0)) == 1.0 else 0 for col in LABEL_COLS]


print("✓ Benchmark metrics defined")

### **Benchmark 1: DDA Agent (Retrieval + Few-Shot + Verification)**Runs the full 3-tool pipeline per image. The verified_analysis JSON is parsed into binary labels for proficiency_index.

In [ ]:
async def benchmark_dda_agent(paths: list, max_images: int = None) -> dict:
    """Benchmark the DDA agent over validation images."""
    agent = DeclarativeDataFlowAgent(
        registry=radiology_registry,
        reasoning_caller=run_medgemma,
        verbose=False,
        system_prompt=RADIOLOGY_SYSTEM_PROMPT,
        goal_few_shot_examples=RADIOLOGY_GOAL_EXAMPLES,
    )

    paths = paths[:max_images] if max_images else paths
    y_true_list, y_pred_list = [], []
    failures = 0

    for img_path in tqdm(paths, desc="DDA Agent", unit="img"):
        gt = get_ground_truth_vector(img_path)
        if gt is None:
            continue

        # Run the full pipeline
        query = f"Analyze this chest X-ray. Patient image: {img_path} Image type: xray"
        result = await agent.run(query, target_key="verified_analysis")

        # Parse the verified analysis
        raw_response = result.get("response", "")
        cxr_dict = parse_cxr_json(raw_response)

        # Also check raw tool results
        if not cxr_dict or len(cxr_dict) < 5:
            # Try to find it in the context keys
            for key in ["verified_analysis", "few_shot_analysis"]:
                if key in result.get("context_keys", []):
                    pass  # Already tried

        if cxr_dict and len(cxr_dict) >= 5:
            pred = cxr_review_to_binary(cxr_dict)
        else:
            # Fallback: use retrieval-only zero-shot vote
            query_vec = embed_query_image(img_path)
            results = faiss_retrieve(query_vec, k=5)
            pred_dict = zero_shot_vote(results)
            pred = [pred_dict[col] for col in LABEL_COLS]
            failures += 1

        y_true_list.append(gt)
        y_pred_list.append(pred)

    y_true = np.array(y_true_list)
    y_pred = np.array(y_pred_list)
    pi = proficiency_index(y_true, y_pred)

    print(f"\n{'='*50}")
    print(f"DDA Agent Proficiency Index: {pi:.4f} ({pi*100:.1f}%)")
    print(f"Images evaluated: {len(y_true_list)} | Parse failures (fallback to retrieval): {failures}")
    print(f"{'='*50}")

    return {"proficiency_index": pi, "y_true": y_true, "y_pred": y_pred,
            "n_images": len(y_true_list), "failures": failures, "agent": "DDA"}


# Run benchmark (use subset for speed — increase for full eval)
dda_results = await benchmark_dda_agent(valid_paths, max_images=200)

### **Benchmark 2: Vanilla MedGemma (No Retrieval, No Harness)**Sends each image directly to MedGemma without any retrieval context or DDA pipeline.

In [ ]:
def benchmark_vanilla_medgemma(paths: list, max_images: int = None) -> dict:
    """Benchmark vanilla MedGemma — image-only, no retrieval."""
    paths = paths[:max_images] if max_images else paths
    y_true_list, y_pred_list = [], []
    failures = 0

    for img_path in tqdm(paths, desc="Vanilla MedGemma", unit="img"):
        gt = get_ground_truth_vector(img_path)
        if gt is None:
            continue

        # Direct image → MedGemma (no retrieval context)
        prompt = CXR_ANALYSIS_PROMPT + "\n\nAnalyze this chest X-ray and return ONLY the JSON."

        content_blocks = [{"type": "text", "text": prompt}]
        if os.path.exists(img_path):
            content_blocks.insert(0, {"type": "image", "image": Image.open(img_path).convert("RGB")})

        try:
            response = run_medgemma(
                messages=[{"role": "user", "content": content_blocks}],
                max_new_tokens=512, temperature=0.1,
            )
            cxr_dict = parse_cxr_json(response)
        except Exception:
            cxr_dict = {}

        if cxr_dict and len(cxr_dict) >= 5:
            pred = cxr_review_to_binary(cxr_dict)
        else:
            pred = [0] * len(LABEL_COLS)  # No fallback for vanilla
            failures += 1

        y_true_list.append(gt)
        y_pred_list.append(pred)

    y_true = np.array(y_true_list)
    y_pred = np.array(y_pred_list)
    pi = proficiency_index(y_true, y_pred)

    print(f"\n{'='*50}")
    print(f"Vanilla MedGemma Proficiency Index: {pi:.4f} ({pi*100:.1f}%)")
    print(f"Images: {len(y_true_list)} | Parse failures: {failures}")
    print(f"{'='*50}")

    return {"proficiency_index": pi, "y_true": y_true, "y_pred": y_pred,
            "n_images": len(y_true_list), "failures": failures, "agent": "Vanilla MedGemma"}


vanilla_results = benchmark_vanilla_medgemma(valid_paths, max_images=200)

### **Benchmark 3: GPT-4o (via OpenAI API, Async)**Calls GPT-4o with each image encoded as base64. Uses `asyncio` for concurrency.

In [ ]:
import openai

# Get OpenAI API key from Kaggle secrets
try:
    openai_key = secrets.get_secret("OPENAI_API_KEY")
    openai_client = openai.AsyncOpenAI(api_key=openai_key)
    print("✓ OpenAI client initialized")
except Exception as e:
    print(f"⚠ OpenAI API key not found: {e}")
    openai_client = None


async def call_gpt4o_single(img_path: str, semaphore: asyncio.Semaphore) -> dict:
    """Call GPT-4o for a single image with concurrency control."""
    async with semaphore:
        try:
            # Encode image as base64
            with open(img_path, "rb") as f:
                b64 = base64.b64encode(f.read()).decode("utf-8")

            response = await openai_client.chat.completions.create(
                model="gpt-4o",
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}", "detail": "low"}},
                        {"type": "text", "text": CXR_ANALYSIS_PROMPT + "\n\nReturn ONLY the JSON object."},
                    ],
                }],
                max_tokens=512,
                temperature=0.1,
            )
            text = response.choices[0].message.content
            return parse_cxr_json(text)
        except Exception as e:
            return {}


async def benchmark_gpt4o(paths: list, max_images: int = None, concurrency: int = 10) -> dict:
    """Benchmark GPT-4o with async concurrency."""
    if openai_client is None:
        print("⚠ OpenAI client not available. Skipping GPT-4o benchmark.")
        return {"proficiency_index": None, "agent": "GPT-4o", "skipped": True}

    paths = paths[:max_images] if max_images else paths
    semaphore = asyncio.Semaphore(concurrency)

    # Prepare tasks
    tasks = []
    gt_list = []
    for img_path in paths:
        gt = get_ground_truth_vector(img_path)
        if gt is None:
            continue
        gt_list.append(gt)
        tasks.append(call_gpt4o_single(img_path, semaphore))

    print(f"Sending {len(tasks)} images to GPT-4o (concurrency={concurrency})...")
    results = []
    # Process in batches for progress tracking
    batch_size = 50
    for i in tqdm(range(0, len(tasks), batch_size), desc="GPT-4o Batches"):
        batch = tasks[i:i+batch_size]
        batch_results = await asyncio.gather(*batch)
        results.extend(batch_results)

    y_true_list, y_pred_list = [], []
    failures = 0
    for gt, cxr_dict in zip(gt_list, results):
        if cxr_dict and len(cxr_dict) >= 5:
            pred = cxr_review_to_binary(cxr_dict)
        else:
            pred = [0] * len(LABEL_COLS)
            failures += 1
        y_true_list.append(gt)
        y_pred_list.append(pred)

    y_true = np.array(y_true_list)
    y_pred = np.array(y_pred_list)
    pi = proficiency_index(y_true, y_pred)

    print(f"\n{'='*50}")
    print(f"GPT-4o Proficiency Index: {pi:.4f} ({pi*100:.1f}%)")
    print(f"Images: {len(y_true_list)} | Parse failures: {failures}")
    print(f"{'='*50}")

    return {"proficiency_index": pi, "y_true": y_true, "y_pred": y_pred,
            "n_images": len(y_true_list), "failures": failures, "agent": "GPT-4o"}


gpt4o_results = await benchmark_gpt4o(valid_paths, max_images=200, concurrency=10)

### **Benchmark 4: Retrieval-Only Baseline (Zero-Shot Vote)**For completeness — the KNN retrieval + majority vote baseline from retrieval-notebook.

In [ ]:
def benchmark_retrieval_only(paths, max_images=None, k=5):
    """Retrieval-only zero-shot vote baseline."""
    paths = paths[:max_images] if max_images else paths
    y_true_list, y_pred_list = [], []

    for path in tqdm(paths, desc=f"Retrieval-Only (k={k})", unit="img"):
        gt = get_ground_truth_vector(path)
        if gt is None:
            continue

        query_vec = embed_query_image(path)
        results = faiss_retrieve(query_vec, k=k)
        pred_dict = zero_shot_vote(results)
        pred = [pred_dict[col] for col in LABEL_COLS]

        y_true_list.append(gt)
        y_pred_list.append(pred)

    y_true = np.array(y_true_list)
    y_pred = np.array(y_pred_list)
    pi = proficiency_index(y_true, y_pred)

    print(f"\nRetrieval-Only (k={k}) Proficiency Index: {pi:.4f} ({pi*100:.1f}%)")
    return {"proficiency_index": pi, "y_true": y_true, "y_pred": y_pred,
            "n_images": len(y_true_list), "agent": f"Retrieval k={k}"}


retrieval_results = benchmark_retrieval_only(valid_paths, max_images=200, k=5)

### **Proficiency Index Comparison**

In [ ]:
import matplotlib.pyplot as plt

# Collect results
all_results = {
    "DDA Agent\n(Retrieval + MedGemma)": dda_results,
    "Vanilla MedGemma\n(No Retrieval)": vanilla_results,
    "Retrieval-Only\n(Zero-Shot Vote)": retrieval_results,
}
if gpt4o_results.get("proficiency_index") is not None:
    all_results["GPT-4o\n(No Retrieval)"] = gpt4o_results

names = list(all_results.keys())
pi_values = [r["proficiency_index"] * 100 for r in all_results.values()]
colors = ["#06b6d4", "#8b5cf6", "#f59e0b", "#f43f5e"][:len(names)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Bar chart
ax = axes[0]
bars = ax.bar(names, pi_values, color=colors, alpha=0.85, width=0.55, edgecolor="white", linewidth=1.5)
ax.set_ylabel("Proficiency Index (%)", fontsize=12, fontweight="bold")
ax.set_title("Proficiency Index Comparison", fontsize=14, fontweight="bold")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.3)
for bar, val in zip(bars, pi_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=12, fontweight="bold")

# Plot 2: Per-label accuracy comparison (DDA vs best baseline)
ax2 = axes[1]
dda_correct = (dda_results["y_true"] == dda_results["y_pred"]).mean(axis=0) * 100
if gpt4o_results.get("y_true") is not None:
    gpt_correct = (gpt4o_results["y_true"] == gpt4o_results["y_pred"]).mean(axis=0) * 100
    baseline_label = "GPT-4o"
else:
    gpt_correct = (vanilla_results["y_true"] == vanilla_results["y_pred"]).mean(axis=0) * 100
    baseline_label = "Vanilla MedGemma"

x = np.arange(len(LABEL_COLS))
width = 0.35
ax2.barh(x - width/2, dda_correct, width, label="DDA Agent", color="#06b6d4", alpha=0.85)
ax2.barh(x + width/2, gpt_correct, width, label=baseline_label, color="#f43f5e", alpha=0.85)
ax2.set_yticks(x)
ax2.set_yticklabels(LABEL_COLS, fontsize=9)
ax2.set_xlabel("Per-Label Accuracy (%)", fontsize=11)
ax2.set_title("Per-Label Accuracy: DDA vs " + baseline_label, fontsize=13, fontweight="bold")
ax2.legend(fontsize=10)
ax2.set_xlim(0, 100)
ax2.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/proficiency_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary table
print("\n" + "=" * 70)
print(f"{'Model':<35} {'Proficiency Index':>20} {'Images':>10}")
print("=" * 70)
for name, result in all_results.items():
    pi = result['proficiency_index']
    n = result['n_images']
    print(f"{name.replace(chr(10), ' '):<35} {pi*100:>18.1f}% {n:>10}")
print("=" * 70)

---### **Streamlit GUI: Gemma Co-Radiologist**The Streamlit app provides a medical workstation interface. To launch:```bashpip install streamlitstreamlit run gemma_co_radiologist_app.py```The GUI requires `athena_dda` to be installed. All SDK imports come from the package.> **Note**: Upload `gemma_co_radiologist_app.py` alongside this notebook to your Kaggle environment.